# 📊 Intelligent Personal Finance Tracker — Data Science Report

**Author:** *Your Name* — 2nd-year Data Science student
**Goal:** Build a system that analyzes personal financial behavior and predicts future spending using statistical methods and machine learning.

This notebook walks through the full pipeline: **data → preprocessing → EDA → insights → ML → evaluation**.


## 1. Imports & data loading

In [ ]:
import sys, os
sys.path.append('..')  # so we can import the `src` package
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid')
from src.data_generator import generate
from src.database import init_db, replace_transactions, load_transactions
from src.preprocessing import add_features, monthly_income_expense
from src import eda, insights, anomaly, recurring, prediction, classifier

# (Re)generate synthetic data to make this notebook self-contained
os.makedirs('../data', exist_ok=True)
os.chdir('..')
df_raw = generate()
init_db(); replace_transactions(df_raw)
df = add_features(load_transactions())
df.head()

## 2. Data overview & cleaning checks

In [ ]:
print('Shape:', df.shape)
print('\nDtypes:'); print(df.dtypes)
print('\nMissing values:'); print(df.isna().sum())
df.describe(include='all').T

## 3. Feature engineering preview
We added: `month`, `day_of_week`, `is_weekend`, `rolling_7d_expense`.

In [ ]:
df[['date','amount','category','type','month','day_of_week','is_weekend','rolling_7d_expense']].head()

## 4. Exploratory Data Analysis

### 4.1 Monthly income vs expense

In [ ]:
monthly = monthly_income_expense(df)
ax = monthly.set_index('month_str')[['income','expense']].plot(kind='bar', figsize=(10,4))
ax.set_ylabel('₹'); ax.set_title('Monthly income vs expense'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
monthly

### 4.2 Spending by category

In [ ]:
cat = eda.category_spend(df)
plt.figure(figsize=(6,6))
plt.pie(cat['amount'], labels=cat['category'], autopct='%1.1f%%'); plt.title('Share of spending by category'); plt.show()
cat

### 4.3 Behavioral patterns — weekend vs weekday & day of week

In [ ]:
print(eda.weekend_vs_weekday(df))
wp = eda.weekday_pattern(df)
sns.barplot(data=wp, x='day_of_week', y='amount'); plt.title('Avg daily expense by day of week'); plt.ylabel('₹'); plt.xticks(rotation=30); plt.show()

### 4.4 Daily expense trend with 7-day rolling mean

In [ ]:
daily = eda.daily_spend(df).set_index('date')
daily['rolling_7d'] = daily['amount'].rolling(7, min_periods=1).mean()
daily.plot(figsize=(11,4)); plt.title('Daily expense + 7-day rolling mean'); plt.ylabel('₹'); plt.show()

## 5. Auto-generated insights
A rule-based engine turns aggregates into natural-language statements.

In [ ]:
for line in insights.generate(df): print('•', line)

## 6. Anomaly detection — Z-score & IQR

In [ ]:
z = anomaly.zscore_anomalies(df, threshold=2.5)
print(f'Z-score outliers: {len(z)}'); z.head()

In [ ]:
iqr = anomaly.iqr_anomalies_per_category(df)
print(f'IQR outliers (per category): {len(iqr)}'); iqr.head()

## 7. Recurring expense detection

In [ ]:
recurring.detect(df)

## 8. ML — next-month expense forecast (Linear Regression vs Moving Average)
Features used: `t` (month index), `prev_month_expense`, `rolling_3m_mean`.
We pick whichever model has the lower MAE on a held-out tail month.

In [ ]:
f = prediction.forecast(df)
print('Forecast for', f.next_month, '→ ₹', round(f.predicted_expense))
print('Chosen model:', f.model_used, '| MAE = ₹', round(f.mae), '| R² =', round(f.r2,3) if f.r2==f.r2 else 'n/a')
hist = f.history.set_index('month_str')
hist.plot(figsize=(10,4), marker='o'); plt.title('Actual vs predicted monthly expense'); plt.ylabel('₹'); plt.show()

## 9. ML — overspend classifier (Logistic Regression)
Label: `expense > income` for the month. Features: previous-month expense, 3-month rolling mean, previous expense ratio.

In [ ]:
res = classifier.predict_overspend(df)
print(res)

## 10. Discussion

**What I learned**
- A clean preprocessing + feature-engineering layer makes every downstream step easier.
- Even a 3-month moving average is a competitive baseline — always compare ML against it.
- Z-score is great for global outliers; IQR per category catches subtler anomalies.
- Pattern-based heuristics are enough to detect recurring bills without ML.

**Limitations**
- Synthetic data: real users have noisier patterns.
- The classifier needs many months of varied data to be reliable.
- Linear regression assumes monotonic trend; a real model might use SARIMA / Prophet.

**Next steps**
- Cluster users into spending types (KMeans on category shares).
- Replace LR with gradient-boosted trees once the dataset is larger.
- Add SMS / bank-statement parsing for real ingestion.
